# 🎓 Classroom Emotion Recognition — DAiSEE Training

> **TRƯỚC KHI CHẠY:** Vào menu **Runtime → Change runtime type → GPU (T4 hoặc A100)**

### Pipeline tổng quan:
1. Cài thư viện & kết nối Kaggle
2. Tải dataset DAiSEE (~15GB)
3. Trích xuất frame từ video
4. Train MobileNetV2 (Transfer Learning)
5. Đánh giá & xuất model `.h5`

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 1 — Kiểm tra GPU & cài thư viện              ║
# ╚══════════════════════════════════════════════════════╝
import subprocess, sys

# Kiểm tra GPU
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    lines = result.stdout.split('\n')
    for l in lines[:4]: print(l)
else:
    print(' KHÔNG có GPU! Vào Runtime → Change runtime type → GPU')

# Cài thêm thư viện cần thiết
!pip install -q kaggle opencv-python-headless scikit-learn seaborn
print(' Thư viện đã sẵn sàng!')

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 2 — Kết nối Kaggle API & Tải Dataset         ║
# ╚══════════════════════════════════════════════════════╝
import os
from google.colab import files

# --- CÁCH 1: Upload file kaggle.json từ máy tính ---
# Lấy kaggle.json: Vào kaggle.com → Account → API → Create New Token
print(' Hãy upload file kaggle.json của bạn:')
uploaded = files.upload()  # Chọn file kaggle.json

# Di chuyển kaggle.json đến vị trí đúng
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
print(' Kaggle API đã được cấu hình!')

# --- Tải dataset DAiSEE ---
print('⏳ Đang tải DAiSEE (~15GB)... (mất khoảng 10-15 phút với GPU T4)')
!kaggle datasets download -d olgaparfenova/daisee -p /content/daisee_raw --unzip
print(' Tải xong!')

# Kiểm tra cấu trúc
!find /content/daisee_raw -maxdepth 3 -type d | head -20

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 3 — Khám phá dataset & đọc CSV Labels        ║
# ╚══════════════════════════════════════════════════════╝
import pandas as pd
import numpy as np
import glob, os

# Tìm file CSV labels
csv_files = glob.glob('/content/daisee_raw/**/*.csv', recursive=True)
print(' Tìm thấy CSV files:')
for f in csv_files: print(' ', f)

# Đọc labels
# DAiSEE có 3 file: Train.csv, Test.csv, Validation.csv
def find_csv(name):
    matches = [f for f in csv_files if name.lower() in f.lower()]
    return matches[0] if matches else None

df_train = pd.read_csv(find_csv('train'))
df_val   = pd.read_csv(find_csv('validat'))
df_test  = pd.read_csv(find_csv('test'))

print(f'\n Train: {len(df_train):,} clips')
print(f'   Val:   {len(df_val):,} clips')
print(f'   Test:  {len(df_test):,} clips')
print(f'\n Columns: {list(df_train.columns)}')
print('\nMẫu dữ liệu:')
df_train.head()

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 4 — Quy đổi nhãn DAiSEE → 4 cảm xúc          ║
# ╚══════════════════════════════════════════════════════╝
import matplotlib.pyplot as plt
import seaborn as sns

EMOTION_NAMES = ['Buồn chán', 'Tập trung', 'Hứng thú', 'Bình thường']

def map_label(row):
    """
    Quy tắc phân loại dựa trên 4 cột DAiSEE (0-3):
      0 = Buồn chán  : Boredom>=2 AND Engagement<=1
      1 = Tập trung  : Engagement>=2 AND Confusion<1
      2 = Hứng thú   : Engagement>=2 AND Confusion>=1 AND Frustration<2
      3 = Bình thường: còn lại
    """
    b = row.get('Boredom', row.get('boredom', 0))
    e = row.get('Engagement', row.get('engagement', 0))
    c = row.get('Confusion', row.get('confusion', 0))
    f = row.get('Frustration', row.get('frustration', 0))

    if b >= 2 and e <= 1:           return 0  # Buồn chán
    if e >= 2 and c >= 1 and f < 2: return 2  # Hứng thú
    if e >= 2:                      return 1  # Tập trung
    return 3                                   # Bình thường

for df, name in [(df_train,'Train'),(df_val,'Val'),(df_test,'Test')]:
    df['emotion_label'] = df.apply(map_label, axis=1)

# Vẽ phân phối nhãn
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (df, name) in zip(axes, [(df_train,'Train'),(df_val,'Val'),(df_test,'Test')]):
    counts = df['emotion_label'].value_counts().sort_index()
    colors = ['#ff6b6b','#4dabf7','#ffd43b','#a9e34b']
    ax.bar([EMOTION_NAMES[i] for i in counts.index], counts.values, color=colors)
    ax.set_title(f'{name} set ({len(df):,} clips)')
    ax.tick_params(axis='x', rotation=20)
    for i, v in enumerate(counts.values):
        ax.text(i, v+5, str(v), ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('/content/label_distribution.png', dpi=120)
plt.show()
print(' Phân phối nhãn OK!')

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 5 — Trích xuất Frame từ Video               ║
# ╚══════════════════════════════════════════════════════╝
import cv2
from pathlib import Path
from tqdm.notebook import tqdm
import shutil

OUTPUT_DIR   = Path('/content/frames')
FRAMES_PER_CLIP = 5      # Lấy 5 frame đều nhau mỗi clip 10s
IMG_SIZE     = (224, 224)

# Tạo thư mục output
for split in ['train','val','test']:
    for i in range(4):
        (OUTPUT_DIR / split / str(i)).mkdir(parents=True, exist_ok=True)

def find_video(clip_name):
    """Tìm đường dẫn video theo tên clip."""
    patterns = [
        f'/content/daisee_raw/**/{clip_name}',
        f'/content/daisee_raw/**/{clip_name}.avi',
    ]
    for p in patterns:
        found = glob.glob(p, recursive=True)
        if found: return found[0]
    return None

def extract_frames(video_path, label, split, clip_id):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total == 0:
        cap.release(); return 0

    indices = np.linspace(0, total-1, FRAMES_PER_CLIP, dtype=int)
    saved = 0
    for i, idx in enumerate(indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret: continue
        frame = cv2.resize(frame, IMG_SIZE)
        out_path = OUTPUT_DIR / split / str(label) / f'{clip_id}_{i}.jpg'
        cv2.imwrite(str(out_path), frame, [cv2.IMWRITE_JPEG_QUALITY, 92])
        saved += 1
    cap.release()
    return saved

stats = {'train': 0, 'val': 0, 'test': 0}

for df, split in [(df_train,'train'), (df_val,'val'), (df_test,'test')]:
    print(f'\n⏳ Đang xử lý {split}...')
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        # Tên clip (cột đầu tiên thường là tên file)
        clip_name = str(row.iloc[0])
        label = int(row['emotion_label'])
        video_path = find_video(clip_name)
        if video_path is None: continue
        n = extract_frames(video_path, label, split, idx)
        stats[split] += n

print('\n Trích xuất xong!')
for s, n in stats.items():
    print(f'   {s}: {n:,} frames')

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 6 — Tạo Data Generator & Class Weights       ║
# ╚══════════════════════════════════════════════════════╝
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf

BATCH_SIZE = 64
NUM_CLASSES = 4

# Data Augmentation cho tập train
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    shear_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3],
    zoom_range=0.1
)

val_gen = ImageDataGenerator(rescale=1./255)

train_flow = train_gen.flow_from_directory(
    str(OUTPUT_DIR / 'train'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

val_flow = val_gen.flow_from_directory(
    str(OUTPUT_DIR / 'val'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_flow = val_gen.flow_from_directory(
    str(OUTPUT_DIR / 'test'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# Tính class weights để xử lý mất cân bằng
labels_arr = train_flow.classes
cw = compute_class_weight('balanced', classes=np.unique(labels_arr), y=labels_arr)
class_weight_dict = dict(enumerate(cw))
# [CẢI TIẾN] Tăng trọng số thủ công cho lớp 0 (Buồn chán) và lớp 3 (Bình thường)
class_weight_dict[0] *= 3.0 
class_weight_dict[3] *= 5.0

print('⚖️  Class weights:', {EMOTION_NAMES[k]: round(v,2) for k,v in class_weight_dict.items()})
print(f'\n Train: {train_flow.samples:,} | Val: {val_flow.samples:,} | Test: {test_flow.samples:,}')

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 7 — Xây dựng mô hình MobileNetV2             ║
# ╚══════════════════════════════════════════════════════╝
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

def build_model(num_classes=4, img_size=(224,224)):
    # Base model: MobileNetV2 pretrained trên ImageNet
    base = MobileNetV2(
        input_shape=(*img_size, 3),
        include_top=False,
        weights='imagenet'
    )
    # Đóng băng toàn bộ base (giai đoạn 1)
    base.trainable = False

    # Thêm classification head
    x = base.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu', kernel_regularizer=l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(128, activation='relu', kernel_regularizer=l2(1e-4))(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=base.input, outputs=out)
    return model, base

model, base_model = build_model(NUM_CLASSES, IMG_SIZE)

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss=tf.keras.losses.CategoricalFocalCrossentropy(alpha=0.25, gamma=2.0),
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# Tóm tắt kiến trúc
total_params = model.count_params()
trainable    = sum(p.numpy().size for p in model.trainable_variables)
print(f' Tổng tham số   : {total_params:,}')
print(f' Tham số trainable: {trainable:,}')
print(f' Tham số frozen   : {total_params - trainable:,}')

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 8 — Training Giai đoạn 1 (Frozen base)       ║
# ╚══════════════════════════════════════════════════════╝
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, TensorBoard
)
import os

os.makedirs('/content/checkpoints', exist_ok=True)

callbacks_phase1 = [
    ModelCheckpoint(
        '/content/checkpoints/best_phase1.h5',
        monitor='val_accuracy', save_best_only=True, mode='max', verbose=1
    ),
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]

print(' Bắt đầu Phase 1 — Training head (base frozen)...')
history1 = model.fit(
    train_flow,
    validation_data=val_flow,
    epochs=15,
    class_weight=class_weight_dict,
    callbacks=callbacks_phase1,
    verbose=1
)
print(f'\n Phase 1 xong! Best val_acc = {max(history1.history["val_accuracy"]):.4f}')

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 9 — Training Giai đoạn 2 (Fine-tuning)       ║
# ╚══════════════════════════════════════════════════════╝

# Mở khóa 40 layer cuối của MobileNetV2 để fine-tune
base_model.trainable = True
for layer in base_model.layers[:-40]:
    layer.trainable = False

trainable_now = sum(p.numpy().size for p in model.trainable_variables)
print(f' Fine-tune {trainable_now:,} tham số')

# Compile lại với LR nhỏ hơn
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss=tf.keras.losses.CategoricalFocalCrossentropy(alpha=0.25, gamma=2.0),
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

callbacks_phase2 = [
    ModelCheckpoint(
        '/content/checkpoints/best_phase2.h5',
        monitor='val_accuracy', save_best_only=True, mode='max', verbose=1
    ),
    EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=4, min_lr=1e-7, verbose=1)
]

print(' Bắt đầu Phase 2 — Fine-tuning 40 layer cuối...')
history2 = model.fit(
    train_flow,
    validation_data=val_flow,
    epochs=30,
    class_weight=class_weight_dict,
    callbacks=callbacks_phase2,
    verbose=1
)
print(f'\n Phase 2 xong! Best val_acc = {max(history2.history["val_accuracy"]):.4f}')

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 10 — Đánh giá & Confusion Matrix             ║
# ╚══════════════════════════════════════════════════════╝
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Tải best model
from tensorflow.keras.models import load_model
best_model = load_model('/content/checkpoints/best_phase2.h5')

# Dự đoán trên test set
print(' Đang dự đoán trên test set...')
test_flow.reset()
y_pred = best_model.predict(test_flow, verbose=1)
y_pred_cls = np.argmax(y_pred, axis=1)
y_true = test_flow.classes[:len(y_pred_cls)]

# Classification Report
print('\n Classification Report:')
print(classification_report(y_true, y_pred_cls, target_names=EMOTION_NAMES))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred_cls)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=EMOTION_NAMES, yticklabels=EMOTION_NAMES)
plt.title('Confusion Matrix — Test Set', fontsize=14)
plt.ylabel('Thực tế'); plt.xlabel('Dự đoán')
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150)
plt.show()

# Vẽ Accuracy/Loss curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
all_acc = history1.history['accuracy'] + history2.history['accuracy']
all_val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
all_loss = history1.history['loss'] + history2.history['loss']
all_val_loss = history1.history['val_loss'] + history2.history['val_loss']
ep = range(1, len(all_acc)+1)
phase1_end = len(history1.history['accuracy'])

axes[0].plot(ep, all_acc, label='Train Acc'); axes[0].plot(ep, all_val_acc, label='Val Acc')
axes[0].axvline(x=phase1_end, color='red', linestyle='--', alpha=0.5, label='Fine-tune Start')
axes[0].set_title('Accuracy'); axes[0].legend()

axes[1].plot(ep, all_loss, label='Train Loss'); axes[1].plot(ep, all_val_loss, label='Val Loss')
axes[1].axvline(x=phase1_end, color='red', linestyle='--', alpha=0.5, label='Fine-tune Start')
axes[1].set_title('Loss'); axes[1].legend()

plt.tight_layout(); plt.savefig('/content/training_curves.png', dpi=150); plt.show()

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 11 — Lưu & Tải model về máy                  ║
# ╚══════════════════════════════════════════════════════╝
from google.colab import files
import zipfile, shutil

# Lưu model
MODEL_OUTPUT = '/content/emotion_model_daisee.h5'
best_model.save(MODEL_OUTPUT)
print(f' Model đã lưu: {MODEL_OUTPUT}')

# Đóng gói tất cả kết quả
with zipfile.ZipFile('/content/emotion_results.zip', 'w') as zf:
    zf.write(MODEL_OUTPUT, 'emotion_model_daisee.h5')
    for fig_name in ['confusion_matrix.png', 'training_curves.png', 'label_distribution.png']:
        path = f'/content/{fig_name}'
        if os.path.exists(path):
            zf.write(path, fig_name)

print(' Đóng gói hoàn tất!')
print('⬇  Đang tải file về máy...')

# Tải model về máy
files.download('/content/emotion_results.zip')
print('\n HOÀN TẤT!')
print(' Đặt file emotion_model_daisee.h5 vào thư mục: TTCS1/models/')
print(' Sau đó chạy: python main.py')

---
## ⚠️ Nếu hết RAM/VRAM giữa chừng

Chạy cell sau để giải phóng bộ nhớ và tiếp tục:

```python
import gc, tensorflow as tf
gc.collect()
tf.keras.backend.clear_session()
```

## 📌 Ghi chú
- **Thời gian ước tính:** Phase 1 ~20 phút + Phase 2 ~60-90 phút (GPU T4)
- **Colab Pro** cho phép dùng A100 → nhanh gấp 3-4 lần
- Model cuối khoảng **~13MB** (MobileNetV2 tối ưu cho mobile)